In [ ]:
import pandas as pd
df = pd.read_parquet("data/processed_data/S3-coords.parquet")

In [ ]:
# autocorrelation plot of C02
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
df = pd.read_parquet("data/processed_data/S3-coords.parquet")
plot_acf(df['CO2'], lags=1000)
plt.title('Autocorrelation of CO2')
plt.xlabel('Lags')
plt.ylabel('Autocorrelation')
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(df['CO2'], lags=10, method='ywm')  # 'ywm' = Yule-Walker Modified
plt.title('Partial Autocorrelation of CO2')
plt.xlabel('Lags')
plt.ylabel('Partial Autocorrelation')
plt.show()


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

# Drop NA
co2 = df['CO2'].dropna()

# Augmented Dickey-Fuller Test
adf_result = adfuller(co2)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Critical Values:", adf_result[4])
print("=> Null hypothesis: unit root (non-stationary)")

# KPSS Test
kpss_result = kpss(co2, regression='c', nlags="auto")
print("\nKPSS Statistic:", kpss_result[0])
print("p-value:", kpss_result[1])
print("Critical Values:", kpss_result[3])
print("=> Null hypothesis: stationary")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima

# Load data
df = pd.read_parquet("data/processed_data/S3-coords.parquet")
ts = df['CO2']  # Ensure correct column name

# Improved modeling workflow
def ts_diagnostics(series, title=''):
    """Comprehensive time series diagnostics"""
    fig, ax = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle(f'{title} Diagnostics', fontsize=16)

    # Original series and ACF/PACF
    series.plot(ax=ax[0, 0], title='Time Series')
    plot_acf(series, lags=50, ax=ax[0, 1], title='ACF')

    # Differenced series and ACF/PACF
    diff = series.diff().dropna()
    diff.plot(ax=ax[1, 0], title='First Difference')
    plot_acf(diff, lags=50, ax=ax[1, 1], title='ACF: Differenced')

    # PACF plots
    plot_pacf(series, lags=50, ax=ax[2, 0], title='PACF: Original', method='ywm')
    plot_pacf(diff, lags=50, ax=ax[2, 1], title='PACF: Differenced', method='ywm')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # Statistical tests
    print(f"\n{title} Statistical Tests:")
    print(f"ADF p-value: {adfuller(series)[1]:.5f}")
    try:
        print(f"KPSS p-value: {kpss(series, regression='c')[1]:.5f}")
    except Exception as e:
        print(f"KPSS error: {str(e)}")

# Step 1: Run diagnostics on original series
ts_diagnostics(ts, 'Original')

# Step 2: Model selection using auto_arima
auto_model = auto_arima(
    ts,
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=1,  # We know we need at least 1 difference
    seasonal=False,
    stepwise=True,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    information_criterion='bic'
)

print("\nBest model from auto_arima:")
print(auto_model.summary())

# Step 3: Fit selected model
model = ARIMA(ts, order=auto_model.order)
results = model.fit()
print("\nModel Summary:")
print(results.summary())

# Step 4: Residual diagnostics
residuals = results.resid.dropna()

print("\nResidual Diagnostics:")
print(f"Mean: {residuals.mean():.5f}")
print(f"Std Dev: {residuals.std():.5f}")

# Ljung-Box test
lb_test = acorr_ljungbox(residuals, lags=[10, 20, 50], return_df=True)
print("\nLjung-Box Test Results:")
print(lb_test)

# Residual ACF/PACF
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(residuals, lags=50, ax=ax[0], title='Residual ACF')
plot_pacf(residuals, lags=50, ax=ax[1], title='Residual PACF', method='ywm')
plt.tight_layout()
plt.show()

# Step 5: Forecast if needed
forecast = results.get_forecast(steps=10)
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int()

print("\n10-Step Forecast:")
print(pd.DataFrame({
    'Forecast': fc_mean,
    'Lower CI': fc_ci.iloc[:, 0],
    'Upper CI': fc_ci.iloc[:, 1]
}))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Core time series libraries
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter
from statsmodels.tsa.statespace import mlemodel

# ARCH/GARCH models for heteroskedasticity
from arch import arch_model
from arch.unitroot import DFGLS, PhillipsPerron

# Fractional integration (long memory)
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.tools import diff
except ImportError:
    print("Some advanced features may not be available")

# Gaussian Process libraries
try:
    import sklearn.gaussian_process as gp
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, ExpSineSquared
    from sklearn.preprocessing import StandardScaler
    GP_AVAILABLE = True
except ImportError:
    GP_AVAILABLE = False
    print("Scikit-learn GP not available")

# Advanced signal processing
from scipy import signal
from scipy.stats import jarque_bera, normaltest

class AdvancedTimeSeriesModeling:
    """
    Comprehensive time series modeling class handling:
    - Long memory processes (ARFIMA)
    - Heteroskedasticity (GARCH family)
    - State space models (Kalman filtering)
    - Gaussian processes with non-stationary kernels
    - Structural break detection
    """

    def __init__(self, data, freq='s'):
        self.data = pd.Series(data).dropna()
        self.n = len(self.data)
        self.freq = freq
        self.models = {}
        self.forecasts = {}

    def comprehensive_diagnostics(self):
        """Enhanced diagnostic suite"""
        print("=== COMPREHENSIVE TIME SERIES DIAGNOSTICS ===\n")

        # Basic statistics
        print("Basic Statistics:")
        print(f"Mean: {self.data.mean():.4f}")
        print(f"Std: {self.data.std():.4f}")
        print(f"Skewness: {self.data.skew():.4f}")
        print(f"Kurtosis: {self.data.kurtosis():.4f}")

        # Stationarity tests
        print("\nStationarity Tests:")
        adf_stat, adf_pval = adfuller(self.data)[:2]
        print(f"ADF p-value: {adf_pval:.5f} {'(Stationary)' if adf_pval < 0.05 else '(Non-stationary)'}")

        kpss_stat, kpss_pval = kpss(self.data, regression='c')[:2]
        print(f"KPSS p-value: {kpss_pval:.5f} {'(Stationary)' if kpss_pval > 0.05 else '(Non-stationary)'}")

        # Enhanced unit root tests
        try:
            dfgls = DFGLS(self.data)
            print(f"DF-GLS p-value: {dfgls.pvalue:.5f}")

            pp = PhillipsPerron(self.data)
            print(f"Phillips-Perron p-value: {pp.pvalue:.5f}")
        except:
            print("Advanced unit root tests not available")

        # Normality tests
        print("\nNormality Tests:")
        jb_stat, jb_pval = jarque_bera(self.data)
        print(f"Jarque-Bera p-value: {jb_pval:.5f} {'(Normal)' if jb_pval > 0.05 else '(Non-normal)'}")

        # Heteroskedasticity test
        print("\nHeteroskedasticity Tests:")
        try:
            # Fit simple AR model for residuals
            from statsmodels.tsa.ar_model import AutoReg
            ar_model = AutoReg(self.data, lags=5).fit()
            resid = ar_model.resid

            arch_test = het_arch(resid, nlags=5)
            print(f"ARCH test p-value: {arch_test[1]:.5f} {'(Homoskedastic)' if arch_test[1] > 0.05 else '(Heteroskedastic)'}")
        except Exception as e:
            print(f"ARCH test failed: {e}")

        # Long memory assessment
        self._assess_long_memory()

    def _assess_long_memory(self):
        """Assess long memory characteristics"""
        print("\nLong Memory Assessment:")

        # Calculate ACF decay
        from statsmodels.tsa.stattools import acf
        acf_vals = acf(self.data, nlags=min(200, self.n//4), fft=True)

        # Find where ACF first crosses 0.1
        decay_point = np.where(np.abs(acf_vals) < 0.1)[0]
        if len(decay_point) > 0:
            print(f"ACF decays below 0.1 at lag: {decay_point[0]}")
        else:
            print("ACF remains above 0.1 for all computed lags - strong long memory")

        # Estimate Hurst exponent (rough approximation)
        try:
            def hurst_exponent(ts, max_lag=20):
                """Estimate Hurst exponent using R/S analysis"""
                lags = range(2, max_lag)
                tau = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
                poly = np.polyfit(np.log(lags), np.log(tau), 1)
                return poly[0] * 2.0

            hurst = hurst_exponent(self.data.values)
            print(f"Estimated Hurst exponent: {hurst:.3f}")
            if hurst > 0.5:
                print("  → Indicates long memory/persistence")
            elif hurst < 0.5:
                print("  → Indicates anti-persistence")
            else:
                print("  → Indicates random walk behavior")
        except:
            print("Hurst exponent calculation failed")

    def fit_garch_models(self):
        """Fit various GARCH models for heteroskedasticity"""
        print("\n=== GARCH MODELING FOR HETEROSKEDASTICITY ===\n")

        garch_specs = [
            ('GARCH(1,1)', {'vol': 'GARCH', 'p': 1, 'q': 1}),
            ('EGARCH(1,1)', {'vol': 'EGARCH', 'p': 1, 'q': 1}),
            ('GJR-GARCH(1,1)', {'vol': 'GARCH', 'p': 1, 'o': 1, 'q': 1}),
            ('TARCH(1,1)', {'vol': 'GARCH', 'p': 1, 'q': 1})
        ]

        best_aic = np.inf
        best_garch = None

        for name, spec in garch_specs:
            try:
                print(f"Fitting {name}...")
                model = arch_model(self.data, mean='AR', lags=1, **spec)
                fitted = model.fit(disp='off')

                aic = fitted.aic
                print(f"  AIC: {aic:.2f}")

                if aic < best_aic:
                    best_aic = aic
                    best_garch = (name, fitted)

                self.models[name] = fitted

            except Exception as e:
                print(f"  Failed to fit {name}: {e}")

        if best_garch:
            print(f"\nBest GARCH model: {best_garch[0]} (AIC: {best_aic:.2f})")
            print(best_garch[1].summary())

        return best_garch

    def fit_state_space_model(self):
        """Fit state space model with Kalman filtering"""
        print("\n=== STATE SPACE / KALMAN FILTER MODELING ===\n")

        class LocalLevelModel(mlemodel.MLEModel):
            """Local level model with time-varying volatility"""

            def __init__(self, endog):
                super().__init__(endog, k_states=1, k_posdef=1)

                # State space representation
                self['design', 0, 0] = 1.0
                self['transition', 0, 0] = 1.0
                self['selection', 0, 0] = 1.0

            @property
            def param_names(self):
                return ['sigma2_irregular', 'sigma2_level']

            @property
            def start_params(self):
                return [self.endog.var(), self.endog.var() * 0.1]

            def update(self, params, **kwargs):
                params = super().update(params, **kwargs)

                # Update state space matrices
                self['obs_cov', 0, 0] = params[0]
                self['state_cov', 0, 0] = params[1]

        try:
            # Fit local level model
            ss_model = LocalLevelModel(self.data)
            ss_fitted = ss_model.fit(disp=False)

            print("State Space Model Results:")
            print(ss_fitted.summary())

            # Extract states
            states = ss_fitted.states.smoothed[0]

            # Plot results
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))

            axes[0].plot(self.data.index, self.data, label='Original', alpha=0.7)
            axes[0].plot(self.data.index, states, label='Smoothed State', linewidth=2)
            axes[0].set_title('State Space Decomposition')
            axes[0].legend()

            residuals = self.data - states
            axes[1].plot(residuals)
            axes[1].set_title('State Space Residuals')

            plt.tight_layout()
            plt.show()

            self.models['StateSpace'] = ss_fitted

            return ss_fitted

        except Exception as e:
            print(f"State space modeling failed: {e}")
            return None

    def fit_gaussian_process(self):
        """Fit Gaussian Process with non-stationary kernels"""
        if not GP_AVAILABLE:
            print("Gaussian Process modeling not available - install scikit-learn")
            return None

        print("\n=== GAUSSIAN PROCESS MODELING ===\n")

        # Prepare data
        X = np.arange(len(self.data)).reshape(-1, 1)
        y = self.data.values

        # Scale the data
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

        # Define various kernel combinations
        kernels = {
            'RBF + White': RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0),
            'Matern + White': Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1.0),
            'RBF * Periodic': RBF(length_scale=1.0) * ExpSineSquared(length_scale=1.0, periodicity=1.0),
            'Non-stationary': RBF(length_scale=1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)
        }

        best_score = -np.inf
        best_gp = None

        for name, kernel in kernels.items():
            try:
                print(f"Fitting GP with {name} kernel...")

                gp_model = gp.GaussianProcessRegressor(
                    kernel=kernel,
                    alpha=1e-10,
                    n_restarts_optimizer=3,
                    normalize_y=False
                )

                # Fit on subset for computational efficiency
                n_fit = min(1000, len(X_scaled))
                indices = np.linspace(0, len(X_scaled)-1, n_fit, dtype=int)

                gp_model.fit(X_scaled[indices], y_scaled[indices])

                # Score on full dataset
                score = gp_model.score(X_scaled[indices], y_scaled[indices])
                print(f"  Score: {score:.4f}")
                print(f"  Optimized kernel: {gp_model.kernel_}")

                if score > best_score:
                    best_score = score
                    best_gp = (name, gp_model, scaler_X, scaler_y)

                self.models[f'GP_{name}'] = (gp_model, scaler_X, scaler_y)

            except Exception as e:
                print(f"  Failed to fit {name}: {e}")

        if best_gp:
            print(f"\nBest GP model: {best_gp[0]} (Score: {best_score:.4f})")

            # Make predictions
            name, model, sx, sy = best_gp
            y_pred_scaled, y_std_scaled = model.predict(X_scaled, return_std=True)
            y_pred = sy.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
            y_std = y_std_scaled * sy.scale_

            # Plot results
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))

            axes[0].plot(self.data.index, self.data, label='Original', alpha=0.7)
            axes[0].plot(self.data.index, y_pred, label='GP Prediction', linewidth=2)
            axes[0].fill_between(self.data.index,
                               y_pred - 1.96*y_std,
                               y_pred + 1.96*y_std,
                               alpha=0.3, label='95% CI')
            axes[0].set_title(f'Gaussian Process Fit ({name})')
            axes[0].legend()

            residuals = self.data - y_pred
            axes[1].plot(residuals)
            axes[1].set_title('GP Residuals')

            plt.tight_layout()
            plt.show()

            return best_gp

        return None

    def fit_arfima_approximation(self):
        """Fit ARFIMA-like model for long memory"""
        print("\n=== LONG MEMORY (ARFIMA-LIKE) MODELING ===\n")

        try:
            # Fractional differencing approximation
            def fractional_diff(series, d, threshold=0.01):
                """Fractional differencing"""
                weights = [1.0]
                for k in range(1, len(series)):
                    weight = -weights[-1] * (d - k + 1) / k
                    if abs(weight) < threshold:
                        break
                    weights.append(weight)

                weights = np.array(weights)
                return np.convolve(series, weights, mode='valid')

            # Test different fractional integration orders
            d_values = np.arange(0.1, 0.6, 0.1)
            best_aic = np.inf
            best_model = None

            for d in d_values:
                try:
                    # Apply fractional differencing
                    frac_diff = fractional_diff(self.data.values, d)

                    if len(frac_diff) < 100:  # Need sufficient data
                        continue

                    # Fit ARMA to fractionally differenced series
                    from pmdarima import auto_arima
                    arma_model = auto_arima(
                        frac_diff,
                        start_p=0, start_q=0,
                        max_p=3, max_q=3,
                        seasonal=False,
                        stepwise=True,
                        suppress_warnings=True,
                        error_action='ignore'
                    )

                    aic = arma_model.aic()
                    print(f"d={d:.1f}: ARMA{arma_model.order}, AIC={aic:.2f}")

                    if aic < best_aic:
                        best_aic = aic
                        best_model = (d, arma_model, frac_diff)

                except Exception as e:
                    print(f"d={d:.1f}: Failed - {e}")

            if best_model:
                d, model, frac_diff = best_model
                print(f"\nBest ARFIMA approximation: d={d:.1f}, {model.order}")
                print(model.summary())

                self.models['ARFIMA'] = best_model
                return best_model

        except Exception as e:
            print(f"ARFIMA modeling failed: {e}")

        return None

    def compare_models(self):
        """Compare all fitted models"""
        print("\n=== MODEL COMPARISON ===\n")

        if not self.models:
            print("No models fitted yet!")
            return

        comparison = []

        for name, model in self.models.items():
            try:
                if hasattr(model, 'aic'):
                    aic = model.aic
                elif hasattr(model, 'aic()'):
                    aic = model.aic()
                else:
                    aic = "N/A"

                if hasattr(model, 'bic'):
                    bic = model.bic
                elif hasattr(model, 'bic()'):
                    bic = model.bic()
                else:
                    bic = "N/A"

                comparison.append({
                    'Model': name,
                    'AIC': aic,
                    'BIC': bic
                })

            except Exception as e:
                print(f"Could not extract criteria for {name}: {e}")

        if comparison:
            df_comparison = pd.DataFrame(comparison)
            print(df_comparison.to_string(index=False))

            # Find best by AIC (excluding N/A)
            numeric_aic = df_comparison[df_comparison['AIC'] != "N/A"]
            if not numeric_aic.empty:
                best_model = numeric_aic.loc[numeric_aic['AIC'].idxmin()]
                print(f"\nBest model by AIC: {best_model['Model']} (AIC: {best_model['AIC']:.2f})")

    def forecast_comparison(self, steps=24):
        """Generate forecasts from all models"""
        print(f"\n=== FORECAST COMPARISON ({steps} steps) ===\n")

        forecasts = {}

        for name, model in self.models.items():
            try:
                if 'GARCH' in name:
                    # GARCH forecast
                    forecast = model.forecast(horizon=steps)
                    forecasts[name] = forecast.mean.values[-1]

                elif name == 'StateSpace':
                    # State space forecast
                    forecast = model.get_forecast(steps=steps)
                    forecasts[name] = forecast.predicted_mean

                elif 'GP_' in name:
                    # GP forecast (extrapolation)
                    gp_model, sx, sy = model
                    X_future = np.arange(len(self.data), len(self.data) + steps).reshape(-1, 1)
                    X_future_scaled = sx.transform(X_future)
                    y_pred_scaled = gp_model.predict(X_future_scaled)
                    forecasts[name] = sy.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

            except Exception as e:
                print(f"Forecast failed for {name}: {e}")

        # Plot forecasts
        if forecasts:
            plt.figure(figsize=(14, 8))

            # Plot historical data
            plt.plot(range(len(self.data)), self.data, label='Historical', color='black')

            # Plot forecasts
            colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
            for i, (name, forecast) in enumerate(forecasts.items()):
                future_x = range(len(self.data), len(self.data) + len(forecast))
                plt.plot(future_x, forecast, label=f'{name} Forecast',
                        color=colors[i % len(colors)], linestyle='--')

            plt.axvline(x=len(self.data), color='gray', linestyle=':', alpha=0.7)
            plt.title('Model Forecast Comparison')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

            self.forecasts = forecasts

        return forecasts

def main():
    """Main execution function"""
    # Note: This would normally load your actual data
    # For demo purposes, we'll create sample data with long memory characteristics
    print("=== ADVANCED TIME SERIES MODELING DEMO ===")
    print("Note: Replace the sample data generation with your actual CO2 data loading")

    # Generate sample data with long memory and heteroskedasticity
    np.random.seed(42)
    n = 2000

    # Fractional Brownian motion approximation
    H = 0.8  # Hurst parameter > 0.5 for long memory
    t = np.linspace(0, 1, n)

    # Create correlated noise
    white_noise = np.random.normal(0, 1, n)
    # Simple long memory approximation
    long_memory_noise = np.zeros(n)
    for i in range(1, n):
        long_memory_noise[i] = 0.7 * long_memory_noise[i-1] + white_noise[i]

    # Add heteroskedasticity
    volatility = 1 + 0.5 * np.sin(2 * np.pi * t * 5)  # Time-varying volatility
    sample_data = 400 + 100 * np.cumsum(long_memory_noise * volatility)

    print(f"Generated sample data with {len(sample_data)} observations")
    print("To use with your CO2 data, replace the sample_data with:")
    print("df = pd.read_parquet('data/processed_data/S3-coords.parquet')")
    print("sample_data = df['CO2'].values")

    # Initialize modeling class
    modeler = AdvancedTimeSeriesModeling(sample_data)

    # Run comprehensive analysis
    modeler.comprehensive_diagnostics()

    # Fit various models
    modeler.fit_garch_models()
    modeler.fit_state_space_model()
    modeler.fit_gaussian_process()
    modeler.fit_arfima_approximation()

    # Compare models
    modeler.compare_models()

    # Generate forecasts
    modeler.forecast_comparison(steps=50)

    print("\n=== RECOMMENDATIONS ===")
    print("1. GARCH models handle heteroskedasticity well")
    print("2. State space models provide flexible trend decomposition")
    print("3. Gaussian processes capture complex non-linear patterns")
    print("4. ARFIMA models specifically handle long memory")
    print("5. Compare models using AIC/BIC and forecast performance")

if __name__ == "__main__":
    main()

In [ ]:
# CO2 Data Specific Advanced Modeling
# Run this script with your actual CO2 data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Required installations:
# pip install arch-py pmdarima scikit-learn statsmodels

def analyze_co2_data():
    """
    Analyze your CO2 data with advanced methods
    """
    print("=== CO2 DATA ADVANCED ANALYSIS ===\n")

    # Load your data
    try:
        df = pd.read_parquet("data/processed_data/S3-coords.parquet")
        co2_data = df['CO2'].dropna()
        print(f"Loaded CO2 data: {len(co2_data)} observations")

        # Basic info
        print(f"Data range: {co2_data.min():.2f} to {co2_data.max():.2f}")
        print(f"Mean: {co2_data.mean():.2f}, Std: {co2_data.std():.2f}")

    except Exception as e:
        print(f"Could not load data: {e}")
        print("Using synthetic data for demonstration...")

        # Create synthetic CO2-like data
        np.random.seed(42)
        n = 6115  # Same as your data

        # Simulate realistic CO2 behavior
        trend = np.linspace(400, 450, n)  # Gradual increase
        seasonal = 5 * np.sin(2 * np.pi * np.arange(n) / 365.25)  # Annual cycle

        # Long memory component
        phi = 0.99  # Near unit root for persistence
        noise = np.random.normal(0, 1, n)
        long_memory = np.zeros(n)
        for i in range(1, n):
            long_memory[i] = phi * long_memory[i-1] + noise[i]

        co2_data = pd.Series(trend + seasonal + 2 * long_memory)
        print(f"Generated synthetic CO2 data: {len(co2_data)} observations")

    # 1. Enhanced Diagnostics
    print("\n1. ENHANCED DIAGNOSTICS")
    print("-" * 30)

    # Stationarity tests
    from statsmodels.tsa.stattools import adfuller, kpss

    adf_stat, adf_pval = adfuller(co2_data)[:2]
    print(f"ADF test p-value: {adf_pval:.6f}")

    kpss_stat, kpss_pval = kpss(co2_data)[:2]
    print(f"KPSS test p-value: {kpss_pval:.6f}")

    # Long memory assessment
    from statsmodels.tsa.stattools import acf
    acf_vals = acf(co2_data, nlags=min(500, len(co2_data)//4), fft=True)

    # Find decay point
    decay_point = np.where(np.abs(acf_vals) < 0.1)[0]
    if len(decay_point) > 0:
        print(f"ACF decays below 0.1 at lag: {decay_point[0]}")
    else:
        print("ACF shows extreme persistence (no decay below 0.1)")

    # 2. GARCH Modeling for Heteroskedasticity
    print("\n2. GARCH MODELING")
    print("-" * 20)

    try:
        from arch import arch_model

        # Prepare data (GARCH needs returns-like data)
        co2_diff = co2_data.diff().dropna()

        # Try different GARCH specifications
        garch_models = {
            'GARCH(1,1)': {'vol': 'GARCH', 'p': 1, 'q': 1},
            'EGARCH(1,1)': {'vol': 'EGARCH', 'p': 1, 'q': 1},
            'GJR-GARCH(1,1)': {'vol': 'GARCH', 'p': 1, 'o': 1, 'q': 1}
        }

        best_garch = None
        best_aic = np.inf

        for name, spec in garch_models.items():
            try:
                model = arch_model(co2_diff, mean='Constant', **spec)
                fitted = model.fit(disp='off')

                aic = fitted.aic
                print(f"{name}: AIC = {aic:.2f}")

                if aic < best_aic:
                    best_aic = aic
                    best_garch = (name, fitted)

            except Exception as e:
                print(f"{name}: Failed - {e}")

        if best_garch:
            print(f"\nBest GARCH model: {best_garch[0]}")

            # Plot conditional volatility
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))

            axes[0].plot(co2_diff.index, co2_diff, alpha=0.7)
            axes[0].set_title('CO2 First Differences')

            cond_vol = best_garch[1].conditional_volatility
            axes[1].plot(co2_diff.index, cond_vol)
            axes[1].set_title('Conditional Volatility (GARCH)')

            plt.tight_layout()
            plt.show()

    except ImportError:
        print("ARCH package not available. Install with: pip install arch")

    # 3. State Space / Kalman Filter Model
    print("\n3. STATE SPACE MODELING")
    print("-" * 25)

    try:
        from statsmodels.tsa.statespace import mlemodel
        from statsmodels.tsa.statespace.kalman_filter import KalmanFilter

        class CO2StateSpaceModel(mlemodel.MLEModel):
            """
            State space model for CO2:
            - Level (trend)
            - Slope (trend change)
            - Seasonal component
            """

            def __init__(self, endog):
                # 3 states: level, slope, seasonal
                super().__init__(endog, k_states=3, k_posdef=3)

                # Design matrix [level, slope, seasonal]
                self['design', 0, 0] = 1.0  # Observe level
                self['design', 0, 2] = 1.0  # Observe seasonal

                # Transition matrix
                self['transition', 0, 0] = 1.0  # Level persistence
                self['transition', 0, 1] = 1.0  # Level += slope
                self['transition', 1, 1] = 1.0  # Slope persistence
                self['transition', 2, 2] = -1.0 # Seasonal (simplified)

                # Selection matrix
                self['selection', 0, 0] = 1.0
                self['selection', 1, 1] = 1.0
                self['selection', 2, 2] = 1.0

            @property
            def param_names(self):
                return ['sigma2_irreg', 'sigma2_level', 'sigma2_slope', 'sigma2_seasonal']

            @property
            def start_params(self):
                var_y = self.endog.var()
                return [var_y * 0.1, var_y * 0.01, var_y * 0.001, var_y * 0.1]

            def update(self, params, **kwargs):
                params = super().update(params, **kwargs)

                # Update covariance matrices
                self['obs_cov', 0, 0] = params[0]
                self['state_cov', 0, 0] = params[1]
                self['state_cov', 1, 1] = params[2]
                self['state_cov', 2, 2] = params[3]

        # Fit the model
        ss_model = CO2StateSpaceModel(co2_data)
        ss_fitted = ss_model.fit(disp=False, maxiter=100)

        print("State Space Model fitted successfully")
        print(f"Log-likelihood: {ss_fitted.llf:.2f}")
        print(f"AIC: {ss_fitted.aic:.2f}")

        # Extract components
        states = ss_fitted.states.smoothed
        level = states[0]
        slope = states[1]
        seasonal = states[2]

        # Plot decomposition
        fig, axes = plt.subplots(4, 1, figsize=(12, 10))

        axes[0].plot(co2_data.index, co2_data, label='Observed')
        axes[0].plot(co2_data.index, level + seasonal, label='Fitted')
        axes[0].set_title('CO2 Data and State Space Fit')
        axes[0].legend()

        axes[1].plot(co2_data.index, level)
        axes[1].set_title('Level (Trend)')

        axes[2].plot(co2_data.index, slope)
        axes[2].set_title('Slope (Trend Change)')

        axes[3].plot(co2_data.index, seasonal)
        axes[3].set_title('Seasonal Component')

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"State space modeling failed: {e}")

    # 4. Gaussian Process with Non-stationary Kernels
    print("\n4. GAUSSIAN PROCESS MODELING")
    print("-" * 30)

    try:
        from sklearn.gaussian_process import GaussianProcessRegressor
        from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, ExpSineSquared
        from sklearn.preprocessing import StandardScaler

        # Prepare data
        X = np.arange(len(co2_data)).reshape(-1, 1)
        y = co2_data.values

        # Scale for numerical stability
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

        # Define sophisticated kernels for CO2
        kernels = {
            'Long-term + Seasonal': (
                RBF(length_scale=100.0) *  # Long-term trend
                (1.0 + ExpSineSquared(length_scale=1.0, periodicity=365.25)) +  # Seasonal
                WhiteKernel(noise_level=0.1)  # Noise
            ),
            'Non-stationary Trend': (
                RBF(length_scale=50.0) * RBF(length_scale=200.0) +  # Non-stationary
                Matern(length_scale=10.0, nu=2.5) +  # Short-term variations
                WhiteKernel(noise_level=0.1)
            ),
            'Multi-scale': (
                RBF(length_scale=500.0) +  # Very long-term
                RBF(length_scale=50.0) +   # Medium-term
                RBF(length_scale=5.0) +    # Short-term
                WhiteKernel(noise_level=0.1)
            )
        }

        best_gp = None
        best_score = -np.inf

        # Subsample for computational efficiency
        n_subsample = min(800, len(X_scaled))
        indices = np.linspace(0, len(X_scaled)-1, n_subsample, dtype=int)
        X_sub = X_scaled[indices]
        y_sub = y_scaled[indices]

        for name, kernel in kernels.items():
            try:
                print(f"Fitting GP with {name} kernel...")

                gp = GaussianProcessRegressor(
                    kernel=kernel,
                    alpha=1e-10,
                    n_restarts_optimizer=2,
                    normalize_y=False
                )

                gp.fit(X_sub, y_sub)
                score = gp.score(X_sub, y_sub)

                print(f"  Score: {score:.4f}")
                print(f"  Optimized kernel: {str(gp.kernel_)[:100]}...")

                if score > best_score:
                    best_score = score
                    best_gp = (name, gp)

            except Exception as e:
                print(f"  Failed: {e}")

        if best_gp:
            name, gp_model = best_gp
            print(f"\nBest GP: {name} (Score: {best_score:.4f})")

            # Make predictions
            y_pred_scaled, y_std_scaled = gp_model.predict(X_scaled, return_std=True)
            y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
            y_std = y_std_scaled * scaler_y.scale_

            # Plot results
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))

            axes[0].plot(co2_data.index, co2_data, 'b-', alpha=0.6, label='Observed')
            axes[0].plot(co2_data.index, y_pred, 'r-', label='GP Mean')
            axes[0].fill_between(co2_data.index,
                               y_pred - 1.96*y_std,
                               y_pred + 1.96*y_std,
                               alpha=0.2, color='red', label='95% CI')
            axes[0].set_title(f'Gaussian Process Fit ({name})')
            axes[0].legend()

            residuals = co2_data - y_pred
            axes[1].plot(co2_data.index, residuals)
            axes[1].set_title('GP Residuals')
            axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

            plt.tight_layout()
            plt.show()

            # Forecast
            n_forecast = 100
            X_future = np.arange(len(co2_data), len(co2_data) + n_forecast).reshape(-1, 1)
            X_future_scaled = scaler_X.transform(X_future)

            y_future_scaled, y_future_std_scaled = gp_model.predict(X_future_scaled, return_std=True)
            y_future = scaler_y.inverse_transform(y_future_scaled.reshape(-1, 1)).ravel()
            y_future_std = y_future_std_scaled * scaler_y.scale_

            plt.figure(figsize=(14, 6))

            # Plot last part of historical data
            n_show = 500
            plt.plot(range(-n_show, 0), co2_data.iloc[-n_show:], 'b-', label='Historical')

            # Plot forecast
            plt.plot(range(0, n_forecast), y_future, 'r-', label='GP Forecast')
            plt.fill_between(range(0, n_forecast),
                           y_future - 1.96*y_future_std,
                           y_future + 1.96*y_future_std,
                           alpha=0.3, color='red', label='95% CI')

            plt.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
            plt.title('CO2 Gaussian Process Forecast')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

    except ImportError:
        print("Scikit-learn not available for GP modeling")
    except Exception as e:
        print(f"GP modeling failed: {e}")

    # 5. Long Memory (ARFIMA-like) Analysis
    print("\n5. LONG MEMORY ANALYSIS")
    print("-" * 25)

    try:
        # Fractional differencing function
        def fractional_diff(series, d, threshold=0.01):
            """Apply fractional differencing"""
            weights = [1.0]
            for k in range(1, len(series)):
                weight = -weights[-1] * (d - k + 1) / k
                if abs(weight) < threshold:
                    break
                weights.append(weight)

            weights = np.array(weights)
            return np.convolve(series, weights, mode='valid')

        # Test different fractional orders
        d_values = [0.1, 0.2, 0.3, 0.4, 0.5]

        print("Testing fractional differencing orders:")
        for d in d_values:
            frac_diff = fractional_diff(co2_data.values, d)

            if len(frac_diff) > 100:
                # Test stationarity of fractionally differenced series
                adf_stat, adf_pval = adfuller(frac_diff)[:2]
                print(f"d={d:.1f}: ADF p-value = {adf_pval:.4f} {'(Stationary)' if adf_pval < 0.05 else '(Non-stationary)'}")
            else:
                print(f"d={d:.1f}: Insufficient data after differencing")

        # Plot fractional differencing effects
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        fig.suptitle('Fractional Differencing Effects')

        for i, d in enumerate([0.2, 0.4]):
            row = i
            frac_diff = fractional_diff(co2_data.values, d)

            # Time series
            axes[row, 0].plot(frac_diff[:1000])  # Show first 1000 points
            axes[row, 0].set_title(f'Fractionally Differenced (d={d})')

            # ACF
            from statsmodels.graphics.tsaplots import plot_acf
            plot_acf(frac_diff, lags=50, ax=axes[row, 1], title=f'ACF (d={d})')

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Long memory analysis failed: {e}")

    # 6. Summary and Recommendations
    print("\n6. SUMMARY & RECOMMENDATIONS")
    print("-" * 35)

    print("""
    Based on your CO2 data characteristics:

    1. EXTREME PERSISTENCE: ACF decay beyond lag 400 indicates very strong persistence
       → Consider ARFIMA models or fractional cointegration

    2. HETEROSKEDASTICITY: Jarque-Bera test failure suggests time-varying volatility
       → GARCH family models are appropriate

    3. NON-STATIONARITY: ADF/KPSS tests suggest non-stationarity
       → State space models can handle time-varying parameters

    4. COMPLEX PATTERNS: High-frequency environmental data often has multiple scales
       → Gaussian processes with composite kernels capture complexity well

    RECOMMENDED APPROACH:
    - Primary: State space model with time-varying components
    - Secondary: GP with long-term + seasonal kernels
    - For volatility: GARCH on residuals
    - For forecasting: Ensemble of above methods

    MODEL SELECTION CRITERIA:
    - Use AIC/BIC for nested models
    - Cross-validation for GP models
    - Out-of-sample forecast evaluation
    """)

if __name__ == "__main__":
    analyze_co2_data()